In [1]:
import re
import requests
import pandas as pd
from pathlib import Path
from time import sleep

In [2]:
def get_kic_from_koi(koi_name):
    """Resolves a KOI string (e.g., 'K00812.01') to a KIC ID using Exo.MAST."""
    try:
        info = requests.get(
            "https://exo.mast.stsci.edu/api/v0.1/exoplanets/identifiers/",
            params={"name": koi_name},
            timeout=30,
        ).json()
        return info.get('keplerID')
    except Exception as e:
        print(f"Error fetching {koi_name}: {e}")
        return None

In [3]:
def get_kepler_fits_urls(kic):
    """Scrapes the MAST archive for long cadence .fits file URLs for a given KIC."""
    if not kic: 
        return []
    
    kic9 = f"{int(kic):09d}"
    url = f"https://archive.stsci.edu/pub/kepler/lightcurves/{kic9[:4]}/{kic9}/"
    
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        # Extract only long cadence light curves
        files = re.findall(r'href="([^"]+_llc\.fits)"', r.text)
        return [url + f for f in files]
    except Exception as e:
        print(f"Error scraping files for KIC {kic}: {e}")
        return []

In [4]:
kois_to_download = [
    "K00812.01", 
    "K00072.01"
]

In [5]:
lines = [
    "#!/usr/bin/env bash",
    "set -euo pipefail",
    ""
]

In [6]:
for koi in kois_to_download:
    # 1. Map KOI to KIC
    kic = get_kic_from_koi(koi)
    
    if kic:
        print(f"Resolved {koi} -> KIC {kic}")
        
        # 2. Get the .fits download links
        urls = get_kepler_fits_urls(kic)
        
        # 3. Append to our curl script lines
        lines.append(f"# {koi} -> KIC {kic}")
        for u in urls:
            lines.append(f"curl -O {u}")
        lines.append("")
    else:
        print(f"WARNING: Could not resolve KIC for {koi}")
        
    # Be polite to the API to avoid timeouts/bans
    sleep(0.2)

Resolved K00812.01 -> KIC 4139816
Resolved K00072.01 -> KIC 11904151


In [7]:
out = Path("download_koi_lcs.sh")
out.write_text("\n".join(lines) + "\n")
out.chmod(0o755)